# Tutorial: SAM 3.1 Video Segmentation with Meta Model API

**Audience:** Python developers who want to segment a video without hosting SAM 3.1.

**Prerequisites:** Python 3.10+, a Jupyter kernel with `requirements-api.txt` installed, a Meta Model API key with access to `sam-3.1`, and a short video URL that Meta can fetch. This route needs no local GPU, PyTorch, or Hugging Face checkpoint download. The [native notebook](sam3_inference.ipynb) covers running Meta's predictor yourself.

**Learning goals:** Send one video and one concept phrase, parse the streamed response, and save mask crops with the frame and object IDs needed to align them with the source video.

The SDK request and parser are covered by offline fixtures in this companion. No paid inference run was made to validate this example, so those checks do not establish account access, segmentation quality, or latency.

## Outline

1. Install the hosted client and load the helper
2. Choose a video and one concept
3. Send an explicit API request
4. Inspect saved masks and their coordinates
5. Group results by object ID

## 1. Set up the CPU client

Run these commands in a terminal from the `SAM-3` folder, using the Python environment selected by your notebook kernel:

```bash
python -m pip install -r requirements-api.txt
python -m pip install jupyterlab ipykernel
jupyter lab sam3_meta_api.ipynb
```

The optional API requirements pin the OpenAI-compatible client, Meta's parser, and Pillow. They are independent of the native CUDA setup. The OpenAI SDK connects to **Meta's** `https://api.meta.ai/v1` endpoint; it uses your Meta key, not an OpenAI key. Create and manage that key through [Meta Model API](https://dev.meta.ai/docs/authentication).

Run this notebook from the folder containing `meta_sam31_video_api.py`.

In [ ]:
import os
from datetime import datetime
from getpass import getpass
from importlib.metadata import version
from pathlib import Path

from IPython.display import display
from PIL import Image

from meta_sam31_video_api import segment_video

for package in ("openai", "meta-sam-parser", "Pillow"):
    print(f"{package}: {version(package)}")

## 2. Choose the input

Start with a short MP4 or MOV clip encoded with H.264, H.265, or MPEG-4. The URL must be directly accessible to Meta; a local path is not a hosted URL. WebM/VP9 is not supported by this API. The documented video limits are 15,000 input frames and 16 tracked objects per frame; check the [current limits](https://dev.meta.ai/docs/sam/segmenting) before a larger run.

Use one concrete noun phrase, such as `person` or `red bicycle`. Comma-separated labels are not a multi-concept request. Use a separate request for each concept.

Each request may incur charges under your Meta account's [pricing and limits](https://dev.meta.ai/docs/pricing-rate-limits). Keep `RUN_HOSTED_REQUEST=False` while reading the notebook. Set it to `True` only after replacing `VIDEO_URL` and choosing a prompt. Each run uses a new output directory to keep results separate.

In [ ]:
VIDEO_URL = ""  # Paste a direct HTTP(S) URL to your short MP4/MOV clip.
TEXT_PROMPT = "person"
OUTPUT_DIR = Path("api-output") / datetime.now().strftime("run-%Y%m%d-%H%M%S-%f")
RUN_HOSTED_REQUEST = False

print(f"Concept: {TEXT_PROMPT}")
print(f"Output directory: {OUTPUT_DIR}")

## 3. Send the request

The helper calls `client.responses.create` with `model="sam-3.1"`, an `input_text` part, and an `input_video` part. It requests `one_bit` masks and passes the event stream to Meta's `parse_responses_stream(..., video_segmentation_format())`. It accepts a result only after completion with no parser diagnostics, then keeps the highest mask revision for each object/frame identity.

The cell below is the only cell that sends an inference request. It reads `MODEL_API_KEY` from your environment or prompts for it without displaying it. Do not paste a key into a saved cell. An authenticated account may still lack access to `sam-3.1`; Meta documents `404 model_not_found` for that case.

Use top-level `await` in Jupyter, where an event loop is already running. The command-line helper uses `asyncio.run` instead. Automatic SDK retries are disabled; rerun deliberately if a request fails. This compact example collects the final result in memory, so keep the first clip short.

In [ ]:
summaries = None
if RUN_HOSTED_REQUEST:
    if not VIDEO_URL.strip():
        raise ValueError("Set VIDEO_URL to a directly accessible video URL first.")
    model_api_key = os.environ.get("MODEL_API_KEY") or getpass("Meta Model API key: ")
    try:
        summaries = await segment_video(
            VIDEO_URL,
            TEXT_PROMPT,
            api_key=model_api_key,
            output_dir=OUTPUT_DIR,
        )
    finally:
        del model_api_key
    print(f"Decoded {len(summaries)} object-frame masks")
    print(f"Saved mask crops and masks.json to {OUTPUT_DIR.resolve()}")
else:
    print("Request skipped. Set RUN_HOSTED_REQUEST=True when ready, then rerun this cell.")

## 4. Read the saved masks

`masks.json` stores each source frame index, object ID, accepted mask revision, mask height and width, foreground-pixel count, half-open box `[left, top, right, bottom]`, and PNG filename. The PNG values are 0 for background and 255 for foreground.

Each PNG is the returned **box-local mask crop**, not a full-frame mask. Keep its box when placing it on the corresponding source frame. Frame numbers can skip frames without a visible match, and IDs can be sparse: do not replace either with a list index. An empty result is a valid no-match outcome.

The preview below shows the crop itself. It is not a rendered tracking video.

In [ ]:
if summaries is None:
    print("No request has run yet.")
elif not summaries:
    print("The request completed with no matching objects.")
else:
    for row in summaries[:5]:
        print(row)
    first = summaries[0]
    with Image.open(OUTPUT_DIR / first["mask_file"]) as mask:
        display(mask.copy())

## 5. Exercise: count masks for each object

Group the returned rows by their explicit `object_id`. How many source frames contain a returned mask for each ID? Why might that count be smaller than the video's frame count?

An object may leave the frame or become occluded. IDs are useful for the returned track, but do not assume perfect identity recovery after a full occlusion or exit and re-entry. Counting returned masks is not an accuracy measurement.

In [ ]:
from collections import defaultdict

frames_by_object = defaultdict(set)
for row in summaries or []:
    frames_by_object[row["object_id"]].add(row["frame"])

for object_id, frames in frames_by_object.items():
    print(f"Object {object_id}: masks in {len(frames)} source frames")

## Pitfalls and extensions

- Hugging Face checkpoint access and hosted API entitlement are separate. A local checkpoint approval does not grant API access.
- For a different concept, choose a fresh output directory and send another request. Do not join concepts with commas.
- A completed empty response is different from a truncated or malformed stream. The helper raises on incomplete results and parser diagnostics.
- Meta's streaming snapshots provide incremental progress, but they are cumulative. The parser retains records and raw text internally, so snapshots alone do not provide bounded-memory processing. Do not concatenate every snapshot.
- To render overlays, decode the corresponding source frames and place each crop using its source-frame bounds. OpenCV can perform this rendering; it does not run the hosted model.

### Primary references

- [Meta: segmenting with prompts and video limits](https://dev.meta.ai/docs/sam/segmenting)
- [Meta: reading segmentation output](https://dev.meta.ai/docs/sam/reading-segmentation)
- [Meta: client libraries](https://dev.meta.ai/docs/sam/client-libraries)
- [Meta parser 0.0.5](https://pypi.org/project/meta-sam-parser/0.0.5/)
- [Meta authentication](https://dev.meta.ai/docs/authentication)
- [Meta pricing and rate limits](https://dev.meta.ai/docs/pricing-rate-limits)